# Day 4A（Colab + NVIDIA API）Agent 可观测性：日志、插件与调试

这份 notebook 适配了：

- **Google Colab**
- **NVIDIA API（OpenAI-compatible endpoint）**
- **ADK + LiteLlm**
- **完整代码 + 详细中文注释**

In [21]:
# Step 1: 安装依赖
!python -m pip install -q -U google-adk litellm openai requests pandas python-dotenv aiosqlite

print("✅ Dependencies installed.")


✅ Dependencies installed.


In [22]:
# Step 2: 读取 NVIDIA API Key
import os
import getpass

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
    if not NVIDIA_API_KEY:
        raise ValueError("Colab Secret 'NVIDIA_API_KEY' is empty.")
    print("✅ Loaded NVIDIA_API_KEY from Colab Secrets.")
except Exception as e:
    print(f"⚠️ Could not read Colab Secret automatically: {e}")
    NVIDIA_API_KEY = getpass.getpass("Please paste your NVIDIA_API_KEY here: ").strip()
    print("✅ Loaded NVIDIA_API_KEY from manual input.")

NVIDIA_API_BASE = "https://integrate.api.nvidia.com/v1"

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
os.environ["OPENAI_API_KEY"] = NVIDIA_API_KEY
os.environ["NVIDIA_API_BASE"] = NVIDIA_API_BASE

print("✅ Environment variables prepared.")
print("NVIDIA_API_BASE =", NVIDIA_API_BASE)


⚠️ Could not read Colab Secret automatically: No module named 'google.colab'
✅ Loaded NVIDIA_API_KEY from manual input.
✅ Environment variables prepared.
NVIDIA_API_BASE = https://integrate.api.nvidia.com/v1


In [ ]:
# ============================================
# Step 3: 初始化日志系统，并清理旧日志
# ============================================

import logging
from pathlib import Path

# ── 1. 定义日志文件路径 ──
LOG_FILE = Path("/workspace/Agent/day4a_observability.log")

# ── 2. 清理旧日志文件 ──
# 每次重新运行 notebook 时，先删除已有的日志文件，避免新旧日志混杂。
# 这在反复调试时特别有用，确保每次看到的都是当前运行的干净日志。
if LOG_FILE.exists():      # 检查文件是否存在
    LOG_FILE.unlink()      # 存在则删除（unlink 是 Path 的删除方法）

# ── 3. 配置日志系统 ──
# 它设置全局默认的日志级别、格式和输出目标。
logging.basicConfig(
    # ── 3.1 日志级别 ──
    # INFO 级别表示：INFO、WARNING、ERROR、CRITICAL 级别的日志会被记录；
    # DEBUG 级别及其以下的信息会被忽略。
    # 可选级别（从低到高）：DEBUG < INFO < WARNING < ERROR < CRITICAL
    level=logging.INFO,
    
    # ── 3.2 日志格式 ──
    # 定义每条日志的输出样式，使用占位符：
    #   %(asctime)s    → 时间戳（如 2026-04-28 10:30:45,123）
    #   %(levelname)s  → 级别名称（如 INFO、ERROR）
    #   %(name)s       → 日志器名称（如 google.adk.agents.llm_agent）
    #   %(message)s    → 具体的日志内容
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    
    # ── 3.3 输出处理器（Handlers）──
    # Handler 决定日志"往哪里写"。这里配置了两个，实现双通道输出：
    handlers=[
        # Handler 1: 写入文件
        # FileHandler 把日志持久化到磁盘文件，便于事后分析、审计和长期保存。
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        
        # Handler 2: 输出到控制台
        # StreamHandler 默认输出到 sys.stderr（即 notebook 的 cell 输出区域），
        # 让你在运行时能实时看到日志滚动。
        logging.StreamHandler(),
    ],
    
    # ── 3.4 强制覆盖 ──
    # force=True 表示：即使 logging 之前已经被配置过（比如某个库内部调用了 basicConfig），
    # 也强制用当前配置覆盖。这在 notebook 环境中很重要，因为多次运行 cell 时
    # logging 可能已有残留配置，不加 force 会导致新配置不生效。
    force=True,
)

# ── 4. 记录一条启动日志 ──
# 用 logging.info() 写入一条 INFO 级别的日志，既验证配置是否生效，
# 又在日志文件中留下一个明确的"系统启动"标记。
logging.info("Logging has been configured successfully.")

# ── 5. 控制台提示 ──
# print() 直接输出到 notebook cell，给用户一个直观的配置成功反馈。
print("✅ Logging configured.") 
print("Log file:", LOG_FILE)


2026-04-28 06:11:45,393 | INFO | root | Logging has been configured successfully.


✅ Logging configured.
Log file: /workspace/Agent/day4a_observability.log


In [24]:
# Step 4: 读取当前 endpoint 可用模型
import requests
import pandas as pd

headers = {"Authorization": f"Bearer {os.environ['NVIDIA_API_KEY']}"}

resp = requests.get(
    f"{os.environ['NVIDIA_API_BASE']}/models",
    headers=headers,
    timeout=30,
)
resp.raise_for_status()

models_payload = resp.json()
model_rows = models_payload.get("data", [])
model_ids = [row.get("id") for row in model_rows if row.get("id")]

print(f"✅ Found {len(model_ids)} models.")
print("First 20 model ids:")
for mid in model_ids[:20]:
    print(" -", mid)

pd.DataFrame(model_rows).head()


✅ Found 135 models.
First 20 model ids:
 - 01-ai/yi-large
 - abacusai/dracarys-llama-3.1-70b-instruct
 - adept/fuyu-8b
 - ai21labs/jamba-1.5-large-instruct
 - aisingapore/sea-lion-7b-instruct
 - baai/bge-m3
 - bigcode/starcoder2-15b
 - bytedance/seed-oss-36b-instruct
 - databricks/dbrx-instruct
 - deepseek-ai/deepseek-coder-6.7b-instruct
 - deepseek-ai/deepseek-v3.1-terminus
 - deepseek-ai/deepseek-v3.2
 - deepseek-ai/deepseek-v4-flash
 - deepseek-ai/deepseek-v4-pro
 - google/codegemma-1.1-7b
 - google/codegemma-7b
 - google/deplot
 - google/gemma-2-2b-it
 - google/gemma-2b
 - google/gemma-3-12b-it


,id,object,created,owned_by
0,01-ai/yi-large,model,735790403,01-ai
1,abacusai/dracarys-llama-3.1-70b-instruct,model,735790403,abacusai
2,adept/fuyu-8b,model,735790403,adept
3,ai21labs/jamba-1.5-large-instruct,model,735790403,ai21labs
4,aisingapore/sea-lion-7b-instruct,model,735790403,aisingapore


In [25]:
# Step 5: 自动选择一个模型
preferred_candidates = [
    "openai/gpt-oss-120b",
    "nvidia/llama-3.3-nemotron-super-49b-v1.5",
    "meta/llama-3.1-70b-instruct",
    "meta/llama-3.1-8b-instruct",
]

NVIDIA_MODEL_NAME = None
for candidate in preferred_candidates:
    if candidate in model_ids:
        NVIDIA_MODEL_NAME = candidate
        break

if NVIDIA_MODEL_NAME is None:
    if not model_ids:
        raise RuntimeError("No models returned by NVIDIA /v1/models.")
    NVIDIA_MODEL_NAME = model_ids[0]

os.environ["NVIDIA_MODEL_NAME"] = NVIDIA_MODEL_NAME
print("✅ Selected model:", NVIDIA_MODEL_NAME)


✅ Selected model: openai/gpt-oss-120b


In [26]:
# Step 6: 做一次纯 API 连通性测试
from openai import OpenAI

client = OpenAI(
    base_url=os.environ["NVIDIA_API_BASE"],
    api_key=os.environ["NVIDIA_API_KEY"],
)

smoke_test = client.chat.completions.create(
    model=os.environ["NVIDIA_MODEL_NAME"],
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "Reply with one short sentence to confirm the NVIDIA API connection is working."},
    ],
    temperature=0.2,
    max_tokens=60,
)

print("✅ Direct NVIDIA API test succeeded.")
print(smoke_test.choices[0].message.content)


✅ Direct NVIDIA API test succeeded.
The NVIDIA API connection is working


In [27]:
# Step 7: 导入 ADK 组件
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.adk.plugins.logging_plugin import LoggingPlugin
from google.adk.plugins.base_plugin import BasePlugin
from google.adk.agents.base_agent import BaseAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest

print("✅ ADK imports succeeded.")


✅ ADK imports succeeded.


In [28]:
# Step 8: 定义论文搜索工具（arXiv，无需额外 key）
import requests
import xml.etree.ElementTree as ET
from typing import Dict, Any, List

ARXIV_API_URL = "http://export.arxiv.org/api/query"
ARXIV_NS = {"atom": "http://www.w3.org/2005/Atom"}

def search_recent_papers(topic: str, max_results: int = 5) -> Dict[str, Any]:
    """
    使用 arXiv API 搜索近期论文标题。

    参数：
        topic: 论文主题，例如 "quantum computing"
        max_results: 最多返回多少篇论文

    返回：
        一个结构化字典，包含论文标题列表和计数。
    """
    logging.info(f"[Tool] search_recent_papers called with topic={topic!r}, max_results={max_results}")

    params = {
        "search_query": f"all:{topic}",
        "start": 0,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }

    response = requests.get(
        ARXIV_API_URL,
        params=params,
        headers={"User-Agent": "colab-adk-observability-demo/1.0"},
        timeout=30,
    )
    response.raise_for_status()

    root = ET.fromstring(response.text)
    entries = root.findall("atom:entry", ARXIV_NS)

    titles = []
    for entry in entries:
        title_node = entry.find("atom:title", ARXIV_NS)
        if title_node is not None and title_node.text:
            titles.append(" ".join(title_node.text.split()))

    result = {
        "status": "success",
        "topic": topic,
        "count": len(titles),
        "papers": titles,
    }

    logging.info(f"[Tool] search_recent_papers returning {len(titles)} titles.")
    return result

tool_preview = search_recent_papers("quantum computing", max_results=3)
print("✅ search_recent_papers tool test result:")
print(tool_preview)


2026-04-28 06:11:46,762 | INFO | root | [Tool] search_recent_papers called with topic='quantum computing', max_results=3
2026-04-28 06:11:49,146 | INFO | root | [Tool] search_recent_papers returning 3 titles.


✅ search_recent_papers tool test result:
{'status': 'success', 'topic': 'quantum computing', 'count': 3, 'papers': ['World-R1: Reinforcing 3D Constraints for Text-to-Video Generation', 'Contracting Tensor Networks with Generalized Belief Propagation', 'A Strongly Polynomial Algorithm for Arctic Auctions']}


In [29]:
# Step 9: 定义一个故意写错的计数工具，便于演示 observability
def count_papers_buggy(papers: List[str]) -> int:
    """
    故意写错的版本：
    参数注解看起来没问题，但函数体把 list 当成了 dict。
    这会在真实运行时触发 TypeError。
    """
    logging.info(f"[Tool] count_papers_buggy received object of type: {type(papers)}")
    return len(papers["titles"])


In [ ]:
# ============================================
# Step 10: 定义一个最小可运行的自定义插件
# ============================================
# 本段代码创建一个 ADK 自定义插件，用于在 Agent 执行的关键节点插入观测逻辑。
# 插件是 ADK 可观测性的"钩子系统"——无需修改 Agent 源码，就能监听和统计内部行为。

# ── 定义自定义插件类 ──
# 继承 BasePlugin 是创建 ADK 插件的标准方式。
# BasePlugin 预定义了一系列生命周期钩子（before/after agent/model/tool），
# 子类只需重写关心的钩子，其他钩子会有默认空实现。
class CountInvocationPlugin(BasePlugin):
    """
    统计 agent 与模型请求次数的简单插件。
    
    设计思路：通过在 before_agent_callback 和 before_model_callback 两个钩子点
    累加计数器，实时追踪 Agent 被调用的频率和底层 LLM 的请求量。
    """
    
    # ── 1. 构造函数 ──
    def __init__(self) -> None:
        # 调用父类构造函数，传入插件名称。
        # 名称用于日志标识和插件管理，建议具有可读性。
        super().__init__(name="count_invocation_plugin")
        
        # ── 1.1 Agent 调用计数器 ──
        # 统计 Agent 一整轮（invocation）被调用的次数。
        # 一次 invocation 可能包含：多次 LLM 调用 + 多次工具调用 + 状态更新。
        self.agent_count = 0
        
        # ── 1.2 LLM 请求计数器 ──
        # 统计底层发送给 LLM API 的请求次数。
        # 注意：一次 Agent invocation 通常触发多次 LLM 请求
        # （如：先请求一次 → 发现需要调工具 → 工具结果回传 → 再请求一次生成最终回复）。
        self.llm_request_count = 0

    # ── 2. Agent 执行前的钩子 ──
    # 在 Agent 的每一轮 invocation 开始时被调用。
    # 此时 Agent 刚刚收到用户新消息，尚未开始任何处理。
    async def before_agent_callback(
        self,
        *,
        agent: BaseAgent,           # 当前被调用的 Agent 实例
        callback_context: CallbackContext,  # 回调上下文，包含 session、用户消息等
    ) -> None:
        self.agent_count += 1
        logging.info(f"[CustomPlugin] Agent invocation count = {self.agent_count}")

    # ── 3. LLM 请求前的钩子 ──
    # 在每次向 LLM API 发送请求之前被调用。
    # 如果一次 Agent 执行中模型需要"思考-工具-再思考"，这个钩子会被触发多次。
    async def before_model_callback(
        self,
        *,
        callback_context: CallbackContext,  # 回调上下文
        llm_request: LlmRequest,    # 即将发送给 LLM 的完整请求对象（含消息、工具定义等）
    ) -> None:
        self.llm_request_count += 1
        logging.info(f"[CustomPlugin] LLM request count = {self.llm_request_count}")

# ── 4. 实例化插件 ──
# 创建插件实例，后续需要注册到 Runner 上才能生效：
#   Runner(..., plugins=[count_plugin])
count_plugin = CountInvocationPlugin()
print("✅ Custom plugin created.")


✅ Custom plugin created.


In [ ]:
# ============================================
# Step 11: 创建一个"有 bug"的研究 Agent
# ============================================
# 本段代码创建一个故意设计为有缺陷的研究助手 Agent，
# 用于演示 ADK 的可观测性能力：如何通过日志追踪 Agent 执行流程、定位故障点。
# 
# 预期故障场景：Agent 按指令调用 count_papers_buggy 时，该工具内部会抛出异常，
# 可观测性系统（日志 + 插件）需要捕获并呈现这个错误。

# ── 创建 LlmAgent 实例 ──
broken_research_agent = LlmAgent(
    # ── 1. Agent 名称 ──
    # 使用 "broken_" 前缀明确标识这是一个"故障注入"的演示 Agent，
    # 与正常 Agent 区分，防止在复杂项目中误用。
    name="broken_research_agent",
    
    # ── 2. 模型配置 ──
    # 直接使用 LiteLlm 内联配置，而非调用工厂函数。
    # 这种写法更底层、更透明，适合教学场景：你能一眼看到 API base、key、模型名。
    model=LiteLlm(
        # model 字符串格式：provider/model_name
        # 这里从环境变量读取 NVIDIA_MODEL_NAME，前面 cell 已设置（如 openai/gpt-oss-120b）
        model=f"openai/{os.environ['NVIDIA_MODEL_NAME']}",
        
        # NVIDIA OpenAI-compatible API 的 endpoint 地址
        api_base=os.environ["NVIDIA_API_BASE"],
        
        # API 认证密钥
        api_key=os.environ["NVIDIA_API_KEY"],
    ),
    
    # ── 3. Agent 描述 ──
    # description 是对 Agent 能力的简短说明，通常在多 Agent 系统中用于路由决策
    # （即：哪个 Agent 适合处理当前请求）。这里明确标注"deliberately broken"用于演示。
    description="A deliberately broken research-paper agent for observability demos.",
    
    # ── 4. 系统指令（核心！）──
    # instruction 是发给 LLM 的系统提示，定义了 Agent 必须严格遵守的执行流程。
    # 这里设计了一个 4 步流水线：
    instruction=(
        "You are a research assistant. "
        
        # Step 1: 调用工具搜索论文
        # 强制要求 Agent 必须先调用 search_recent_papers 工具获取数据
        "Follow these steps strictly: "
        "1. Call search_recent_papers. "
        
        # Step 2: 解析工具返回结果
        # 告诉 Agent 从工具输出中提取 papers 列表，为下一步做准备
        "2. Extract the papers list from the tool result. "
        
        # Step 3: 调用计数工具（⭐ 故障注入点）
        # 明确要求 Agent 把上一步的 papers 列表传给 count_papers_buggy。
        # 这个工具内部被故意写了一个 bug（如：类型错误、索引越界、空指针等），
        # 执行到这里会抛出异常。
        "3. Call count_papers_buggy on that papers list. "
        
        # Step 4: 汇总返回
        # 要求 Agent 把论文标题和计数结果一起返回给用户。
        # 但由于 Step 3 会崩溃，这一步实际上走不到——除非有错误恢复机制。
        "4. Return both the titles and the count. "
        
        # 强调纪律：防止 LLM "偷懒"跳过计数步骤直接猜数字
        "Do not skip the counting step."
    ),
    
    # ── 5. 工具列表 ──
    # Agent 可调用的外部工具。这里挂载了两个函数：
    #   - search_recent_papers: 正常工具，返回论文列表
    #   - count_papers_buggy:   故障工具，内部有 bug，用于演示错误追踪
    tools=[search_recent_papers, count_papers_buggy],
)

print("✅ Broken agent created.")


✅ Broken agent created.


In [32]:
# Step 12: 创建 Runner，并挂上 LoggingPlugin + CustomPlugin
broken_runner = InMemoryRunner(
    agent=broken_research_agent,
    plugins=[
        LoggingPlugin(),
        count_plugin,
    ],
)

print("✅ Runner configured with LoggingPlugin + CustomPlugin.")


2026-04-28 06:11:49,248 | INFO | google_adk.google.adk.plugins.plugin_manager | Plugin 'logging_plugin' registered.
2026-04-28 06:11:49,248 | INFO | google_adk.google.adk.plugins.plugin_manager | Plugin 'count_invocation_plugin' registered.


✅ Runner configured with LoggingPlugin + CustomPlugin.


In [33]:
# ============================================
# Step 13: 运行有 bug 的 Agent
# ============================================
# 本段代码执行"故障注入测试"：故意运行一个包含 bug 工具的 Agent，
# 验证可观测性系统（日志 + 异常捕获）能否正确记录和呈现错误。
# 
# 核心教学点：在 Agent 工程中，"失败"不是终点，而是观测的起点。
# 好的可观测性系统让你能精准回答：什么失败了？在哪一步？为什么？

# ── 1. 打印测试声明 ──
# 在控制台输出明确的预期，让阅读者知道：
#   - 我们正在运行的是 broken agent
#   - 失败是"计划内的"，不是意外
#   - 目的是检验日志捕获能力
print("🚀 Running broken agent...")
print("📌 Expected outcome: this run should fail, so that we can inspect the logs.\n")

# ── 2. 用 try/except 包裹执行 ──
try:
    # ── 2.1 触发 Agent 执行 ──
    # run_debug() 是 Runner 提供的调试运行方法，返回完整的执行结果或抛出异常。
    # 与 run_async() 不同，run_debug() 通常会把内部错误直接抛出（而非包装在事件里），
    # 适合用于"预期失败"的测试场景。
    # 
    # 用户请求："查找最近关于量子计算的论文，并统计数量"
    # 这个请求会迫使 Agent 按 instruction 的 4 步走，最终触发 Step 3 的 bug。
    _ = await broken_runner.run_debug(
        "Find recent papers on quantum computing and count how many there are."
    )
    
    # ── 2.2 意外成功分支 ──
    # 如果执行到了这里，说明 bug 没有触发——这反而是异常的！
    # 可能原因：LLM "偷懒"跳过了计数步骤、或者 bug 工具被意外修复了。
    print("⚠️ Unexpected: the broken agent did not fail.")

# ── 3. 捕获并分析异常 ──
# 当 count_papers_buggy 内部抛出异常时，ADK 会把它包装后向上传递。
# 这里捕获异常不是为了"吞掉错误"，而是为了：
#   1. 向用户证明可观测性系统确实捕获到了故障
#   2. 打印异常类型和消息，作为事后分析的起点
except Exception as e:
    print("✅ The broken agent failed exactly as expected.")
    
    # type(e).__name__: 异常类名，如 TypeError、ValueError、ToolError 等
    print("Captured exception type:", type(e).__name__)
    
    # str(e): 异常的具体描述信息，包含错误原因和上下文
    print("Captured exception message:", str(e))


2026-04-28 06:11:49,280 | INFO | root | [CustomPlugin] Agent invocation count = 1
2026-04-28 06:11:49,282 | INFO | root | [CustomPlugin] LLM request count = 1


🚀 Running broken agent...
📌 Expected outcome: this run should fail, so that we can inspect the logs.


 ### Created new session: debug_session_id

User > Find recent papers on quantum computing and count how many there are.
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-4580f680-f7d3-49cf-bb7a-21b1ecc9da03
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]    User ID: debug_user_id
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: broken_research_agent
[logging_plugin]    User Content: text: 'Find recent papers on quantum computing and count how many there are.'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-4580f680-f7d3-49cf-bb7a-21b1ecc9da03
[logging_plugin]    Starting Agent: broken_research_agent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: broken_research_agent
[logging_plugin]    Invocation ID: e-4580f680-f7d3-49cf-bb7a-21b1ecc9da03
[logging_plugin] 🧠 LLM RE

2026-04-28 06:11:49,283 | INFO | LiteLLM | 
LiteLLM completion() model= openai/gpt-oss-120b; provider = openai
2026-04-28 06:11:51,255 | INFO | root | [Tool] search_recent_papers called with topic='quantum computing', max_results=5


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: broken_research_agent
[logging_plugin]    Content: text: 'We must follow developer instructions: call search_recent_papers with topic "quantum computing". Then extract list, then call count_papers_buggy on that list. Then return both titles and count.

But n...' | function_call: search_recent_papers
[logging_plugin]    Token Usage - Input: 370, Output: 219
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 4f179fe9-bdf4-4402-9ac0-0110bf91ea49
[logging_plugin]    Author: broken_research_agent
[logging_plugin]    Content: text: 'We must follow developer instructions: call search_recent_papers with topic "quantum computing". Then extract list, then call count_papers_buggy on that list. Then return both titles and count.

But n...' | function_call: search_recent_papers
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['search_recent_papers']
broken_research_agent > We must follow developer i

2026-04-28 06:11:52,365 | INFO | root | [Tool] search_recent_papers returning 5 titles.
2026-04-28 06:11:52,368 | INFO | root | [CustomPlugin] LLM request count = 2


[logging_plugin] 🔧 TOOL COMPLETED
[logging_plugin]    Tool Name: search_recent_papers
[logging_plugin]    Agent: broken_research_agent
[logging_plugin]    Function Call ID: chatcmpl-tool-9d1b2885bd2a4852
[logging_plugin]    Result: {'status': 'success', 'topic': 'quantum computing', 'count': 5, 'papers': ['World-R1: Reinforcing 3D Constraints for Text-to-Video Generation', 'Contracting Tensor Networks with Generalized Belief Propagation', 'A Strongly Polynomial Algorithm for Arctic Auctions', 'Semiclassical phases of charged s...}
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 2144a493-0698-48cd-8675-79668b2ef7b8
[logging_plugin]    Author: broken_research_agent
[logging_plugin]    Content: function_response: search_recent_papers
[logging_plugin]    Final Response: False
[logging_plugin]    Function Responses: ['search_recent_papers']
[logging_plugin] 🧠 LLM REQUEST
[logging_plugin]    Model: openai/openai/gpt-oss-120b
[logging_plugin]    Agent: broken_research_agent
[lo

2026-04-28 06:11:52,370 | INFO | LiteLLM | 
LiteLLM completion() model= openai/gpt-oss-120b; provider = openai
2026-04-28 06:11:53,596 | INFO | root | [Tool] count_papers_buggy received object of type: <class 'list'>


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: broken_research_agent
[logging_plugin]    Content: text: 'Now extract list, call count_papers_buggy with papers list.' | function_call: count_papers_buggy
[logging_plugin]    Token Usage - Input: 703, Output: 120
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 0a6fd1f7-bbe7-41fc-ba5b-e936b1e693e7
[logging_plugin]    Author: broken_research_agent
[logging_plugin]    Content: text: 'Now extract list, call count_papers_buggy with papers list.' | function_call: count_papers_buggy
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['count_papers_buggy']
broken_research_agent > Now extract list, call count_papers_buggy with papers list.
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: count_papers_buggy
[logging_plugin]    Agent: broken_research_agent
[logging_plugin]    Function Call ID: chatcmpl-tool-acaf85895c34c280
[logging_plugin]    Arguments: {'papers': ['World-R1: Reinf

In [36]:
# Step 14: 查看日志文件最后 200 行
from pathlib import Path

log_path = Path("/workspace/Agent/day4a_observability.log")
if not log_path.exists():
    raise FileNotFoundError(f"Log file not found: {log_path}")

log_text = log_path.read_text(encoding="utf-8")
log_lines = log_text.splitlines()

print("✅ Showing the last 200 log lines:\n")
for line in log_lines[-200:]:
    print(line)


✅ Showing the last 200 log lines:

2026-04-28 06:11:45,393 | INFO | root | Logging has been configured successfully.
2026-04-28 06:11:46,762 | INFO | root | [Tool] search_recent_papers called with topic='quantum computing', max_results=3
2026-04-28 06:11:49,146 | INFO | root | [Tool] search_recent_papers returning 3 titles.
2026-04-28 06:11:49,248 | INFO | google_adk.google.adk.plugins.plugin_manager | Plugin 'logging_plugin' registered.
2026-04-28 06:11:49,248 | INFO | google_adk.google.adk.plugins.plugin_manager | Plugin 'count_invocation_plugin' registered.
2026-04-28 06:11:49,280 | INFO | root | [CustomPlugin] Agent invocation count = 1
2026-04-28 06:11:49,282 | INFO | root | [CustomPlugin] LLM request count = 1
2026-04-28 06:11:49,283 | INFO | LiteLLM | 
LiteLLM completion() model= openai/gpt-oss-120b; provider = openai
2026-04-28 06:11:51,255 | INFO | root | [Tool] search_recent_papers called with topic='quantum computing', max_results=5
2026-04-28 06:11:52,365 | INFO | root | [T

In [37]:
# Step 15: 写一个正确的计数工具，修复 bug
def count_papers_fixed(papers: List[str]) -> int:
    """正确版本：直接返回列表长度。"""
    logging.info(f"[Tool] count_papers_fixed received object of type: {type(papers)}")
    return len(papers)

print("✅ Fixed counting tool created.")


✅ Fixed counting tool created.


In [38]:
# Step 16: 创建修复后的 agent
fixed_research_agent = LlmAgent(
    name="fixed_research_agent",
    model=LiteLlm(
        model=f"openai/{os.environ['NVIDIA_MODEL_NAME']}",
        api_base=os.environ["NVIDIA_API_BASE"],
        api_key=os.environ["NVIDIA_API_KEY"],
    ),
    description="A fixed research-paper agent for observability demos.",
    instruction=(
        "You are a research assistant. "
        "Follow these steps strictly: "
        "1. Call search_recent_papers. "
        "2. Extract the papers list from the tool result. "
        "3. Call count_papers_fixed on that papers list. "
        "4. Return both the titles and the count in clear natural language."
    ),
    tools=[search_recent_papers, count_papers_fixed],
)

fixed_runner = InMemoryRunner(
    agent=fixed_research_agent,
    plugins=[
        LoggingPlugin(),
        CountInvocationPlugin(),
    ],
)

print("✅ Fixed agent and runner created.")


2026-04-28 06:12:54,553 | INFO | google_adk.google.adk.plugins.plugin_manager | Plugin 'logging_plugin' registered.
2026-04-28 06:12:54,554 | INFO | google_adk.google.adk.plugins.plugin_manager | Plugin 'count_invocation_plugin' registered.


✅ Fixed agent and runner created.


In [39]:
# Step 17: 再运行一次，这次应该成功
print("🚀 Running fixed agent...\n")

fixed_response = await fixed_runner.run_debug(
    "Find recent papers on quantum computing and count how many there are."
)

print("\n✅ Fixed agent run completed.")
print("Returned object type:", type(fixed_response))


2026-04-28 06:12:57,076 | INFO | root | [CustomPlugin] Agent invocation count = 1
2026-04-28 06:12:57,077 | INFO | root | [CustomPlugin] LLM request count = 1


🚀 Running fixed agent...


 ### Created new session: debug_session_id

User > Find recent papers on quantum computing and count how many there are.
[logging_plugin] 🚀 USER MESSAGE RECEIVED
[logging_plugin]    Invocation ID: e-01706547-2e2c-40e7-b89c-88cdc008f8bd
[logging_plugin]    Session ID: debug_session_id
[logging_plugin]    User ID: debug_user_id
[logging_plugin]    App Name: InMemoryRunner
[logging_plugin]    Root Agent: fixed_research_agent
[logging_plugin]    User Content: text: 'Find recent papers on quantum computing and count how many there are.'
[logging_plugin] 🏃 INVOCATION STARTING
[logging_plugin]    Invocation ID: e-01706547-2e2c-40e7-b89c-88cdc008f8bd
[logging_plugin]    Starting Agent: fixed_research_agent
[logging_plugin] 🤖 AGENT STARTING
[logging_plugin]    Agent Name: fixed_research_agent
[logging_plugin]    Invocation ID: e-01706547-2e2c-40e7-b89c-88cdc008f8bd
[logging_plugin] 🧠 LLM REQUEST
[logging_plugin]    Model: openai/openai/gpt-oss-120b
[logging_plugin]   

2026-04-28 06:12:57,078 | INFO | LiteLLM | 
LiteLLM completion() model= openai/gpt-oss-120b; provider = openai
2026-04-28 06:12:58,713 | INFO | root | [Tool] search_recent_papers called with topic='quantum computing', max_results=5


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: fixed_research_agent
[logging_plugin]    Content: text: 'We need to follow steps: call search_recent_papers with topic "quantum computing". Use default max_results (5). Then extract papers list from result. Then call count_papers_fixed with that list. Then ...' | function_call: search_recent_papers
[logging_plugin]    Token Usage - Input: 329, Output: 92
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: aeed3441-9c9f-483a-9772-8ed51fa70bf6
[logging_plugin]    Author: fixed_research_agent
[logging_plugin]    Content: text: 'We need to follow steps: call search_recent_papers with topic "quantum computing". Use default max_results (5). Then extract papers list from result. Then call count_papers_fixed with that list. Then ...' | function_call: search_recent_papers
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['search_recent_papers']
fixed_research_agent > We need to follow steps: call 

2026-04-28 06:13:01,897 | INFO | root | [Tool] search_recent_papers returning 5 titles.
2026-04-28 06:13:01,900 | INFO | root | [CustomPlugin] LLM request count = 2


[logging_plugin] 🔧 TOOL COMPLETED
[logging_plugin]    Tool Name: search_recent_papers
[logging_plugin]    Agent: fixed_research_agent
[logging_plugin]    Function Call ID: chatcmpl-tool-9de0b470e4cbd6dd
[logging_plugin]    Result: {'status': 'success', 'topic': 'quantum computing', 'count': 5, 'papers': ['World-R1: Reinforcing 3D Constraints for Text-to-Video Generation', 'Contracting Tensor Networks with Generalized Belief Propagation', 'A Strongly Polynomial Algorithm for Arctic Auctions', 'Semiclassical phases of charged s...}
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: f7f8ca79-dc88-4f30-9803-0a55fb2f26cb
[logging_plugin]    Author: fixed_research_agent
[logging_plugin]    Content: function_response: search_recent_papers
[logging_plugin]    Final Response: False
[logging_plugin]    Function Responses: ['search_recent_papers']
[logging_plugin] 🧠 LLM REQUEST
[logging_plugin]    Model: openai/openai/gpt-oss-120b
[logging_plugin]    Agent: fixed_research_agent
[loggi

2026-04-28 06:13:01,902 | INFO | LiteLLM | 
LiteLLM completion() model= openai/gpt-oss-120b; provider = openai
2026-04-28 06:13:02,988 | INFO | root | [Tool] count_papers_fixed received object of type: <class 'list'>
2026-04-28 06:13:02,990 | INFO | root | [CustomPlugin] LLM request count = 3


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: fixed_research_agent
[logging_plugin]    Content: text: 'Now call count_papers_fixed with papers list.' | function_call: count_papers_fixed
[logging_plugin]    Token Usage - Input: 535, Output: 115
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 17cea30c-b216-4b46-a3c2-e53a2093cde7
[logging_plugin]    Author: fixed_research_agent
[logging_plugin]    Content: text: 'Now call count_papers_fixed with papers list.' | function_call: count_papers_fixed
[logging_plugin]    Final Response: False
[logging_plugin]    Function Calls: ['count_papers_fixed']
fixed_research_agent > Now call count_papers_fixed with papers list.
[logging_plugin] 🔧 TOOL STARTING
[logging_plugin]    Tool Name: count_papers_fixed
[logging_plugin]    Agent: fixed_research_agent
[logging_plugin]    Function Call ID: chatcmpl-tool-9303f2995b6c59bd
[logging_plugin]    Arguments: {'papers': ['World-R1: Reinforcing 3D Constraints for Text-to-Video Genera

2026-04-28 06:13:02,991 | INFO | LiteLLM | 
LiteLLM completion() model= openai/gpt-oss-120b; provider = openai


[logging_plugin] 🧠 LLM RESPONSE
[logging_plugin]    Agent: fixed_research_agent
[logging_plugin]    Content: text: 'Here are the recent papers I found on **quantum computing**:

1. World‑R1: Reinforcing 3D Constraints for Text‑to‑Video Generation  
2. Contracting Tensor Networks with Generalized Belief Propagation ...'
[logging_plugin]    Token Usage - Input: 670, Output: 119
[logging_plugin] 📢 EVENT YIELDED
[logging_plugin]    Event ID: 23eca400-a5a6-4a5d-bcbe-1fdcc9ad47e8
[logging_plugin]    Author: fixed_research_agent
[logging_plugin]    Content: text: 'Here are the recent papers I found on **quantum computing**:

1. World‑R1: Reinforcing 3D Constraints for Text‑to‑Video Generation  
2. Contracting Tensor Networks with Generalized Belief Propagation ...'
[logging_plugin]    Final Response: True
fixed_research_agent > Here are the recent papers I found on **quantum computing**:

1. World‑R1: Reinforcing 3D Constraints for Text‑to‑Video Generation  
2. Contracting Tensor Networks wit

## 小结

如果你顺利跑完前 17 格，你已经完整体验了 observability 的核心流程：

1. 先让系统跑起来  
2. 制造一个真实业务 bug  
3. 通过日志和插件观察行为  
4. 定位根因  
5. 修复 bug  
6. 再次运行验证